In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

# Warnings ko ignore karne ke liye
warnings.filterwarnings('ignore')

# Preprocessing aur data splitting
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Linear model (Perceptron)
from sklearn.linear_model import Perceptron  # Used for simple linear classification tasks.

# Model evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Deep Learning (TensorFlow/Keras)
from tensorflow.keras.models import Sequential  # Sequential lets you build a neural network layer-by-layer in Keras.

from tensorflow.keras.layers import Dense        # Dense makes the final predictions
from tensorflow.keras.layers import Conv2D       # Conv2D extracts features
from tensorflow.keras.layers import Flatten      # Flatten reshapes them
from tensorflow.keras.layers import MaxPooling2D # MaxPooling2D reduces size
from tensorflow.keras.layers import Dropout      # Dropout prevents overfitting
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers
from tensorflow import keras

In [ ]:
df = pd.read_csv('/Workspace/Users/sadiqrangrej10@gmail.com/train.txt', sep=';', header=None, names=['text', 'emotion'])
df.head()
df.isnull().sum()
unique_emotions = df['emotion'].unique()
print(unique_emotions)
# Create a dictionary to map emotions to numbers
emotion_numbers = {}
i = 0
for emo in unique_emotions:
    emotion_numbers[emo] = i
    i += 1

df['emotion'] = df['emotion'].map(emotion_numbers)
df
df['text'] = df['text'].apply(lambda x : x.lower())
import string
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))
df['text'] = df['text'].apply(remove_punctuation)

def remove_numbers(text):
    new = ""
    for i in text:
        if not i.isdigit():
            new = new + i 
    return new
df['text'] = df['text'].apply(remove_numbers)
import re

def remove_urls(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', text)

df['text'] = df['text'].apply(remove_urls)
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new
df['text'] = df['text'].apply(remove_emojis)


In [ ]:
%pip install nltk

In [ ]:
import nltk 
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
print(len(stop_words))
stop_words
df.loc[1]['text']
def remove(txt):
    words = txt.split()
    cleaned = []
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
    return " ".join(cleaned)
df['text'] = df['text'].apply(remove)
df.loc[1]['text']
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

pred_nb = nb_model.predict(X_test_bow)
print(accuracy_score(y_test,pred_nb))
tfidf_vectorizer = TfidfTransformer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_bow)
X_test_tfidf = tfidf_vectorizer.transform(X_test_bow)
nb_model_tfidf = MultinomialNB()
nb_model_tfidf.fit(X_train_tfidf, y_train)
pred_nb_tfidf = nb_model_tfidf.predict(X_test_tfidf)
print(accuracy_score(y_test,pred_nb_tfidf))
from sklearn.linear_model import LogisticRegression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
pred_lr = lr_model.predict(X_test_tfidf)
print(lr_model.score(X_test_tfidf, y_test))